Code from class for querying data

In [3]:
import requests
import pandas as pd
import geopandas as gpd
import time

def fetch_acs5_tracts(year: int, state_fips: str, county_fips: str, variables: list[str], api_key: str | None = None) -> pd.DataFrame:
    """Fetch ACS 5-year data for all tracts in a county."""
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    get_vars = ["NAME"] + variables
    params = {
        "get": ",".join(get_vars),
        "for": "county:*",
        "in": f"state:{state_fips}"
    }

    if api_key:
        params["key"] = api_key

    r = requests.get(base, params=params, timeout=60)

    if r.status_code != 200:
        print(f"  [!] Failed to fetch {year} data. Check variables or API status.")
        return pd.DataFrame()

    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df["GEOID"] = df["state"] + df["county"]

    for v in variables:
        df[v] = pd.to_numeric(df[v], errors="coerce")

    return df

YEARS_ACS = [2018, 2019, 2020, 2021, 2022, 2023]
YEAR_TIGER = 2023
STATE_FIPS = "06"
COUNTY_FIPS = "*"

vars_list = [
    "B01003_001E",
    "B19013_001E",
    "B25064_001E",
    "B17001_001E",
    "B17001_002E"
]

all_years_data = []

print("Fetching ACS data across multiple years...")
for y in YEARS_ACS:
    print(f"  Downloading {y}...")
    temp_df = fetch_acs5_tracts(y, STATE_FIPS, COUNTY_FIPS, vars_list)

    if not temp_df.empty:
        temp_df["Data_Year"] = y
        all_years_data.append(temp_df)

    time.sleep(1)


acs_ca_multiyear = pd.concat(all_years_data, ignore_index=True)

tiger_url = f"https://www2.census.gov/geo/tiger/TIGER{YEAR_TIGER}/COUNTY/tl_{YEAR_TIGER}_us_county.zip"

print(f"Downloading {YEAR_TIGER} TIGER shapefiles for US Counties...")
counties_us = gpd.read_file(tiger_url)

print("Filtering down to California counties...")
counties_ca = counties_us[counties_us["STATEFP"] == STATE_FIPS].copy()

print("Merging time-series data with polygons...")
gdf = counties_ca.merge(acs_ca_multiyear, on="GEOID", how="inner")

print("Reprojecting CRS...")
gdf = gdf.to_crs(epsg=3857)

print("\nDone! Data ready for mapping.")
gdf[['GEOID', 'Data_Year', 'B19013_001E', 'geometry']].head(10)


Fetching ACS data across multiple years...
Filtering down to California counties...
Merging time-series data with polygons...
Reprojecting CRS...

Done! Data ready for mapping.


,GEOID,Data_Year,B19013_001E,geometry
0,06091,2018,48125,"POLYGON ((-13420217.727 4794807.849, -13420247..."
1,06091,2019,52148,"POLYGON ((-13420217.727 4794807.849, -13420247..."
2,06091,2020,52103,"POLYGON ((-13420217.727 4794807.849, -13420247..."
3,06091,2021,56152,"POLYGON ((-13420217.727 4794807.849, -13420247..."
4,06091,2022,61108,"POLYGON ((-13420217.727 4794807.849, -13420247..."
5,06091,2023,60000,"POLYGON ((-13420217.727 4794807.849, -13420247..."
6,06067,2018,63902,"POLYGON ((-13518629.055 4615586.93, -13518641...."
7,06067,2019,67151,"POLYGON ((-13518629.055 4615586.93, -13518641...."
8,06067,2020,70684,"POLYGON ((-13518629.055 4615586.93, -13518641...."
9,06067,2021,76422,"POLYGON ((-13518629.055 4615586.93, -13518641...."


In [6]:
gdf.shape

(348, 28)

Should have 348 rows (58 counties x 6 years)

In [ ]:
gdf.to_csv('census_california.csv')

Write to CSV